# Colab süreç denetimi — CPU yazılım sınaması

Python 3 / **CPU** / Runtime Version **2026.07** seçin.
Bu notebook yalnızca üretilmiş, küçük bir alt süreç çalıştırır; Torch yüklemez.
Yerel çalışma diski kullanılır. Paket kurulumu, ağ isteği ve Drive bağlantısı yoktur.
Süre, veri bölümü ve dosya sınırları çekirdeğe uygulanır; RAM izlenir.
**Kesin cgroup RAM/swap sınırı sağlanmaz.**
Kernel tepe RAM değeri süreç açılışını içerir.
Aynı çalışma dizininde deneme tekrarlanamaz; kesilmiş kayıtları koruyun.
Bu yazılımda native ölçüm/eğitim komutu yoktur. Eski deneyleri yeniden çalıştırmaz.


In [ ]:
import base64
import hashlib
import json
import os
import sys
from pathlib import Path

SOURCE_SHA256 = '104fcbbc46f1fcee36ca83ca42e1e267e64eace9ed7616d5b0b767fe45fa7d92'
BASE_COMMIT = '2f41bfe2e95febcaab80f80ab4875c72073c3a32'
SOURCE_B64 = (
    '...'
    'Cn0K'
)


def install_sources(raw, root):
    allowed = set(
        ('core/__init__.py', 'core/colab_artifacts.py', 'core/colab_execution.py', 'LICENSE')
    )
    if len(raw) > 262144 or hashlib.sha256(raw).hexdigest() != SOURCE_SHA256:
        raise ValueError("source package hash/size mismatch")
    value = json.loads(raw)
    if (set(value) != {"schema", "base_commit", "files"}
            or value["schema"] != "colab-process-source-v1"
            or value["base_commit"] != BASE_COMMIT or set(value["files"]) != allowed):
        raise ValueError("source package schema mismatch")
    prepared = {}
    for name, entry in value["files"].items():
        if set(entry) != {"utf8", "sha256"}:
            raise ValueError("source entry schema mismatch")
        content = entry["utf8"].encode("utf-8")
        if len(content) > 65536 or hashlib.sha256(content).hexdigest() != entry["sha256"]:
            raise ValueError("source file hash/size mismatch")
        prepared[name] = content
    if any(p.is_symlink() for p in (root, *root.parents)):
        raise ValueError("source directory symlink")
    root.mkdir(mode=0o700)
    (root / "core").mkdir(mode=0o700)
    for name, content in prepared.items():
        fd = os.open(root / name, os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_NOFOLLOW, 0o600)
        with os.fdopen(fd, "wb") as stream:
            stream.write(content)
    return root


if any(name == "core" or name.startswith("core.") for name in sys.modules):
    raise RuntimeError("use a fresh CPU kernel; a core module is already loaded")
source_root = install_sources(base64.b64decode(SOURCE_B64, validate=True),
                             Path("/content/spatial-colab-process-source-20261106"))
sys.path.insert(0, str(source_root))
print("Verified generated process QA sources:", SOURCE_SHA256)


In [ ]:
from core import colab_artifacts as artifacts
from core import colab_execution as service

protocol = source_root / "generated-process-contract.json"
output = source_root / "generated-process-result"
limits = service.Limits(10, 128 * artifacts.MIB, 8 * 1024**3,
                        8 * artifacts.MIB, artifacts.MIB,
                        address_bytes=256 * artifacts.MIB, closing_seconds=1)
service.freeze(source_root, protocol, "generated_qa", {"action": "success", "value": 0}, limits)
result = service.run(source_root, protocol, output)
service.audit(source_root, protocol, output)
print(json.dumps(result, sort_keys=True, indent=2))
